# Result Analysis

*too: description*

In this notebook I analyse the results produced by running the experiment.

In [ ]:
import logging
import os.path

import datetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objs as go
from plotly.subplots import make_subplots

from metrics.mae import MAE
from metrics.coverage_rate import CoverageRate, RollingCoverageRate, CoveredDimensions, RollingCoverageRateByFeature
from metrics.interval_width import MeanIntervalWidth, RollingMeanIntervalWidth, RollingMeanIntervalWidthByFeature

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(filename)s:%(lineno)s %(funcName)s() %(message)s')

In [ ]:
eval_fn = lambda f: f()  # self-invoking function decorator to keep the namespace clean

In [ ]:
experiment_dir_name = os.getcwd().split('/')[-1]
print(f'experiment_dir_name: {experiment_dir_name}')
os.chdir('../../../')
# os.getcwd()

%pwd

In [ ]:
experiment_group_dir = Path('assets/experimental_results') / experiment_dir_name
weight_functions_and_corrections = {
    'sol_icp_bf_exp_b0.007': 'sol_icp_bf_exponential_b0.007.json',
    'sol_icp_bf_sc_c200_s50': 'sol_icp_bf_soft_cutoff_c200_s50.json',
    'sol_icp_bf_lin': 'sol_icp_bf_linear.json',
    'sol_icp_bf_cst': 'sol_icp_bf_constant.json',
    'ol_icp_bf_cst': 'ol_icp_bf_constant.json'
}
univariate_and_multivariate_ds = {
    'multv': 'elec2_ds_multivariate.json',
}

weight_function_names = ['Exponential', 'Soft Cutoff', 'Linear', 'Constant', 'CF-RNN']

fig_target_path = Path(
    '/Users/filip.schlembach/Documents/unimaas_local/phd_repos/um_phd_24_copa_springer_speical_issue')
fig_exp_name = experiment_dir_name[:6]

experiment_dirs = []
for icp_setting, icp_params_file in weight_functions_and_corrections.items():
    for ds_setting, ds_params_file in univariate_and_multivariate_ds.items():
        experiment_dirs.append(f'{ds_setting}_{icp_setting}')

output dirs

In [ ]:
def now() -> str:
    return datetime.datetime.now().strftime('%y%m%d_%H%M')

## MAE

In [ ]:
mae_results = {exp_dir: MAE.load(
    os.path.join(os.path.join(experiment_group_dir, exp_dir + '/summary'), MAE.snake_name())) for exp_dir in
    experiment_dirs}

print('underlying model mae on test set:', mae_results)

## Coverage Rate

In [ ]:
@eval_fn
def _():
    cr_results = [CoverageRate.load(
        os.path.join(os.path.join(experiment_group_dir, exp_dir + '/summary'), CoverageRate.snake_name())) for exp_dir
        in
        experiment_dirs]
    plt.rcParams.update({'font.size': 16})
    CoverageRate.comparative_plot(cr_results, weight_function_names + ['Target'],
                                  save_path=fig_target_path / f'{fig_exp_name}_cr.svg',
                                  fig_size=(5, 5))
    cr_df = pd.DataFrame([[*cr_results[i][1]] for i in range(len(cr_results))], columns=1 - cr_results[0][0],
                         index=weight_function_names).T
    cr_df = cr_df.round(decimals=3)
    # cr_df.to_csv(table_out_dir / f'cr_{now()}.csv')
    print(cr_df)

### Rolling Coverage Rate

In [ ]:
@eval_fn
def _():
    rcr_results = [RollingCoverageRate.load(
        os.path.join(os.path.join(experiment_group_dir, exp_dir + '/summary'), RollingCoverageRate.snake_name())) for
        exp_dir in experiment_dirs]
    RollingCoverageRate.comparative_plot(rcr_results, weight_function_names, 0.2, fig_size=(6, 3),
                                         save_path=fig_target_path / f'{fig_exp_name}_rcr.svg',
                                         dotted_v_line_idxs=[267, 534])

### Rolling Coverage Rate by Feature

In [ ]:
@eval_fn
def _():
    rcrf_results = [RollingCoverageRateByFeature.load(
        os.path.join(os.path.join(experiment_group_dir, exp_dir + '/summary'),
                     RollingCoverageRateByFeature.snake_name()))
        for
        exp_dir in experiment_dirs]
    RollingCoverageRateByFeature.comparative_plot(rcrf_results, weight_function_names, ['\n$y_{1:800,1}$', '\n$y_{1:800,2}$'], 0.2,
                                                  fig_size=(6, 5.5),
                                                  save_path=fig_target_path / f'{fig_exp_name}_rcrf.svg',
                                                  alpha_prime=0.1,
                                                  dotted_v_line_idxs=[267, 534])

## Interval width

In [ ]:
@eval_fn
def _():
    miw_results = [MeanIntervalWidth.load(
        os.path.join(os.path.join(experiment_group_dir, exp_dir + '/summary'), MeanIntervalWidth.snake_name())) for
        exp_dir
        in
        experiment_dirs]
    MeanIntervalWidth.comparative_plot(miw_results, weight_function_names,
                                       save_path=fig_target_path / f'{fig_exp_name}_miw.svg',
                                       fig_size=(5, 5))

    miw_df = pd.DataFrame([[*miw_results[i][1]] for i in range(len(miw_results))], columns=1 - miw_results[0][0],
                          index=weight_function_names).T
    miw_df = miw_df.round(decimals=3)
    # miw_df.to_csv(table_out_dir / f'miw_{now()}.csv')
    print(miw_df)

In [ ]:
@eval_fn
def _():
    rmiw_results = [RollingMeanIntervalWidth.load(
        os.path.join(os.path.join(experiment_group_dir, exp_dir + '/summary'), RollingMeanIntervalWidth.snake_name()))
        for
        exp_dir in experiment_dirs]

    RollingMeanIntervalWidth.comparative_plot(rmiw_results, weight_function_names, 0.2, fig_size=(6, 3),
                                              save_path=fig_target_path / f'{fig_exp_name}_rmiw.svg',
                                              y_lim=(0,3),
                                              dotted_v_line_idxs=[267, 534])

In [ ]:
@eval_fn
def _():
    frmiw_results = [RollingMeanIntervalWidthByFeature.load(
        os.path.join(os.path.join(experiment_group_dir, exp_dir + '/summary'),
                     RollingMeanIntervalWidthByFeature.snake_name()))
        for
        exp_dir in experiment_dirs]

    RollingMeanIntervalWidthByFeature.comparative_plot(frmiw_results, weight_function_names, ['$y_{1:800,1}$', '$y_{1:800,2}$'], 0.2,
                                                       fig_size=(6, 5.5),
                                                       save_path=fig_target_path / f'{fig_exp_name}_rmiwf.svg',
                                                       y_lim=(0, 3),
                                                       dotted_v_line_idxs=[267, 534])

## Visualisation of Prediction Regions

In [ ]:
def plot_test_set_y(trial_path: Path, alpha: str) -> None:
    test_y = np.load(trial_path / 'test_y.npy')
    test_y_alpha = np.load(trial_path / f'test_{alpha}.npy')
    test_y_model = np.load(trial_path / f'test_model.npy')
    test_y = test_y.reshape((2400, 2))
    test_y_alpha = test_y_alpha.reshape((2400, 2, 2))
    test_y_model = test_y_model.reshape((2400, 2))

    fig = make_subplots(rows=2, cols=1)

    for f in range(2):
        y_upper = test_y_alpha[:, f, 1]
        y_lower = test_y_alpha[:, f, 0]
        y_band = np.concatenate((y_upper, y_lower[::-1]), axis=None)
        x = [i for i in range(2400)]

        fig.add_trace(go.Scatter(
            x=x,
            y=test_y[:, f],
            line=dict(color='rgb(0,0,100)'),
            mode='lines',
            name=f'y {f + 1}'
        ), row=f + 1, col=1)
        fig.add_trace(go.Scatter(
            x=x,
            y=test_y_model[:, f],
            line=dict(color='rgb(0,100,80)'),
            mode='lines',
            name=f'model {f + 1}'
        ), row=f + 1, col=1)
        fig.add_trace(go.Scatter(
            x=x + x[::-1],  # x, then x reversed
            y=y_band,  # upper, then lower reversed
            fill='toself',
            fillcolor='rgba(0,100,80,0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo="skip",
            showlegend=False
        ), row=f + 1, col=1)
    fig.show()

In [ ]:
plot_test_set_y(experiment_group_dir / 'multv_sol_icp_bf_exp_b0.007/trial02',
                '0.2')

## Completeness

Since there have been issues with certain aspects of the execution in the past, I'm checking if all relevant files have been saved.
If there are missing files the experiment will have to be re-run.

In [ ]:
from pathlib import Path

experiment_group_dir = Path(experiment_group_dir)
experiments = {}
for experiment_dir in experiment_group_dir.iterdir():
    if experiment_dir.is_dir():
        experiments[experiment_dir.name] = {}
        for trial_dir in experiment_dir.iterdir():
            if trial_dir.is_dir():
                experiments[experiment_dir.name][trial_dir.name] = {}
                for result_file in trial_dir.iterdir():
                    experiments[experiment_dir.name][trial_dir.name][result_file.name] = result_file.stat().st_ctime
                # experiments[experiment_dir.name][trial_dir.name]['_'] = np.NaN

        experiments[experiment_dir.name] = pd.DataFrame(experiments[experiment_dir.name])  # == 1
        experiments[experiment_dir.name] = experiments[experiment_dir.name].sort_index()
        experiments[experiment_dir.name] = experiments[experiment_dir.name].reindex(
            sorted(experiments[experiment_dir.name].columns), axis=1)

__Success__: All relevant files are present. The experiment does not need to be re-run.